**Author**: JW Debelius (justine.debelius@jhu.edu)<br>
**Project**: 017-Arg<br>
**Enviroment**: <br>
**Creation Date**: July 2021<br>
**Update Date**: November 2025

This notebbok will take the metadata file and sync with the feature tables.

In [1]:
import os
import shutil

import biom
import numpy as np
import pandas as pd
import scipy
import skbio

from qiime2 import Artifact, Metadata, Visualization

import qiime2.plugins.boots.actions as q2_boots
import qiime2.plugins.diversity.actions as q2_diversity
import qiime2.plugins.feature_table.actions as q2_feature_table
import qiime2.plugins.gemelli.actions as q2_gemelli
import qiime2.plugins.longitudinal.actions as q2_longitudinal
import qiime2.plugins.taxa.actions as q2_taxa

/Users/jwd/miniconda3/envs/arg-amplicon-2025.7/lib/python3.10/site-packages/q2_demux/_summarize/_visualizer.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# Sample metadata

In [2]:
meta = pd.read_csv('../data/metadata/analysis_sample_meta.tsv',
                   sep='\t',
                   dtype=str)
meta.set_index('sample-id', inplace=True)

In [3]:
working_meta = meta.copy()
float_cols = ['age', 
              'days_since_intervention',
              'timepoint',
              'gtdb_depth']
working_meta[float_cols] = working_meta[float_cols].astype(float)
working_meta = working_meta.loc[working_meta['gtdb_depth'] > int(1e6)].copy()
working_meta_q2 = Metadata(working_meta)

And then we'll read in the taxonomy tables and import everything associated into QIIME 2.

In [4]:
species_table = biom.load_table('../data/Stag_output/gtdb_species.biom')
genome_table = biom.load_table('../data/Stag_output/gtdb_genome.biom')

In [5]:
species_q2 = Artifact.import_data('FeatureTable[Frequency]', species_table, biom.Table)
genome_q2 = Artifact.import_data('FeatureTable[Frequency]', genome_table, biom.Table)

In [6]:
species_m_q2, = q2_feature_table.filter_samples(species_q2, metadata=Metadata(meta))
genome_m_q2, = q2_feature_table.filter_samples(genome_q2, metadata=Metadata(meta))

In [7]:
genome_taxa = genome_table.metadata_to_dataframe(axis='observation').apply(
    lambda x: ';'.join(x), axis=1)
genome_taxa.index.set_names('Feature ID', inplace=True)
genome_taxa.name = "Taxon"
genome_taxa_q2 = Artifact.import_data('FeatureData[Taxonomy]', genome_taxa, pd.Series)

In [8]:
genome_tree_q2 = Artifact.load('../data/Stag_output/matched_tree_gtdb.qza')

In [9]:
os.makedirs('final-data/02.tables/', exist_ok=True)
species_m_q2.save('final-data/02.tables/matched-species-gtdb.qza')
genome_m_q2.save('final-data/02.tables/matched-genome-gtdb.qza')
genome_taxa_q2.save('final-data/02.tables/matched-taxonomy-gtdb.qza')
genome_tree_q2.save('final-data/02.tables/matched_tree_gtdb.qza')

'final-data/02.tables/matched_tree_gtdb.qza'

We're going to use q2-boots (an update) to do multiple rarefaction and perform diversity calculations. We'll collapse the species level data for alpha diversity and taxonomic beta diversity and do UniFrac on the genome table.

In [10]:
%%capture
resampled_genome_q2, = q2_boots.resample(genome_m_q2, 
                                         sampling_depth=int(1e6), 
                                         n=25,
                                         replacement=False)
resampled_species_q2 = {id_: q2_taxa.collapse(table_q2, 
                                              taxonomy=genome_taxa_q2, 
                                              level=7).collapsed_table
                        for id_, table_q2 in resampled_genome_q2.items()}

In [11]:
%%capture
alpha_spec_q2 = {metric: {i: q2_diversity.alpha(table=table_q2, metric=metric).alpha_diversity
                          for i, table_q2 in resampled_species_q2.items()}
                 for metric in ['observed_features', 'shannon', 'simpson']}
alpha_average_q2 = {
    metric: q2_boots.alpha_average(alphas, 
                                   average_method='median').average_alpha_diversity
    for metric, alphas in alpha_spec_q2.items()
    }

In [12]:
os.makedirs('final-data/04.alpha-diveristy/', exist_ok=True)
for metric, alpha_q2 in alpha_average_q2.items():
    alpha_q2.save(f'final-data/04.alpha-diveristy/{metric}_1M_avg.qza')

In [13]:
%%capture
phylo_metrics = ['unweighted_unifrac', 'weighted_unifrac']
taxon_metrics = ['braycurtis', 'jaccard']
betat_spec_q2 = {
    metric: {i: q2_diversity.beta(table=table_q2, metric=metric).distance_matrix
             for i, table_q2 in resampled_species_q2.items()}
    for metric in taxon_metrics
}
betap_spec_q2 = {
    metric: {i: q2_diversity.beta_phylogenetic(table=table_q2, 
                                               phylogeny=genome_tree_q2, 
                                               metric=metric).distance_matrix
             for i, table_q2 in resampled_genome_q2.items()}
    for metric in phylo_metrics
}
betat_spec_q2.update(betap_spec_q2)
beta_average_q2 = {
    metric: q2_boots.beta_average(dms_q2, 
                                  average_method='non-metric-median').average_distance_matrix
    for metric, dms_q2 in betat_spec_q2.items()
    }

In [14]:
beta_average_q2

{'braycurtis': <artifact: DistanceMatrix uuid: 42fee2a3-6294-49cd-a082-584c8b9c6af2>,
 'jaccard': <artifact: DistanceMatrix uuid: f439273b-c8cf-4625-bf98-d8b3c567e97c>,
 'unweighted_unifrac': <artifact: DistanceMatrix uuid: 957f7c75-f661-4af7-9c4c-7aa75166c54c>,
 'weighted_unifrac': <artifact: DistanceMatrix uuid: 12b2b8e3-0b74-485c-b56a-7bf4c3d1faf5>}

In [15]:
os.makedirs('final-data/06.beta-diveristy/', exist_ok=True)
for metric, dm_q2 in beta_average_q2.items():
    dm_q2.save(f'final-data/06.beta-diveristy/{metric}_1M_avg_dm.qza')

We'll look at the longitudinal change in alpha diveristy from baseline

In [16]:
%%capture
renames = {
    'shannon': 'shannon_entropy',
    'simpson': alpha_average_q2['simpson'].view(pd.Series).name,
}
alpha_baseline_diffs = {
    metric: q2_longitudinal.first_differences(
        working_meta_q2.merge(art.view(Metadata)),
        metric=renames.get(metric, metric),
        state_column='timepoint',
        individual_id_column='host_subject_id',
        baseline=0,
    ).first_differences
    for metric, art in alpha_average_q2.items()
}
alpha_baseline_diffs['depth'] = \
    q2_longitudinal.first_differences(
        working_meta_q2,
        metric='gtdb_depth',
        state_column='timepoint',
        individual_id_column='host_subject_id',
        baseline=0,
    ).first_differences

for metric, alpha in alpha_baseline_diffs.items():
    alpha.save(f'final-data/04.alpha-diveristy/{metric}_1M_avg_from_baseline.qza')

And we'll do the same with beta diversity.

In [17]:
%%capture
from_baseline = {
    metric: q2_longitudinal.first_distances(
        distance_matrix=dm_q2,
        metadata=working_meta_q2,
        state_column='timepoint',
        individual_id_column='host_subject_id',
        baseline=0,
    ).first_distances
    for metric, dm_q2 in beta_average_q2.items()
}
for metric, dm_q2 in from_baseline.items():
    dm_q2.save(f'final-data/06.beta-diveristy/{metric}_1M_avg_from_baseline.qza')

We're adding DEICODE and PhyloDEICODE because they're awesome.

To get there, we'll start by filtering the feature table. Our filtering critiera are that we want features present withh a relative abundance of at least $1\times10^5$ in at least 2 samples in at least 10% of participants.

In [18]:
keep_filt = genome_m_q2.view(biom.Table)
keep_filt.add_metadata(meta.to_dict(orient='index'), axis='sample')
keep_filt.norm(axis='sample', inplace=True)
keep_filt.transform(lambda v, id_, md: v > 5e-5, axis='sample')
keep_filt = keep_filt.collapse(lambda id_, md: md['host_subject_id'],
                               norm=False,
                               axis='sample')
keep_filt.filter(lambda v, id_, md: (v > 2).mean() >= 0.1, 
                 axis='observation')
keep_filt = keep_filt.to_dataframe().sparse.to_dense()
keep_filt.index.set_names('id', inplace=True)

And then we'll filter the features

In [19]:
table_filt_q2, = q2_feature_table.filter_features(table=genome_m_q2,
                                                  metadata=Metadata(keep_filt)
                                                  )
os.makedirs('final-data/08.rpca', exist_ok=True)
table_filt_q2.save('final-data/08.rpca/filtered-table.qza')

'final-data/08.rpca/filtered-table.qza'

I wnat to filter the data to the baseline to calculate the rPCA.

In [20]:
table_filt_bl_q2, = q2_feature_table.filter_samples(table=table_filt_q2,
                                                   metadata=working_meta_q2,
                                                   where='[timepoint] = 0')
deicode_bl_biplot_q2, deicdoe_bl_dm_q2 = \
    q2_gemelli.rpca(table_filt_bl_q2, n_components=3)
phylo_rpca_bl_results = \
    q2_gemelli.phylogenetic_rpca_with_taxonomy(table=table_filt_bl_q2,
                                                  phylogeny=genome_tree_q2,
                                                  taxonomy=genome_taxa_q2.view(Metadata),
                                                  n_components=3
                                                 )

In [21]:
os.makedirs('final-data/05.rpca/baseline_rpca', exist_ok=True)
table_filt_bl_q2.save('final-data/05.rpca/baseline_rpca/filt_table.qza')
deicode_bl_biplot_q2.save('final-data/05.rpca/baseline_rpca/biplot.qza')
deicdoe_bl_dm_q2.save('final-data/05.rpca/baseline_rpca/distance-matrix.qza')

os.makedirs('final-data/08.rpca/baseline_phylo_rpca', exist_ok=True)
phylo_rpca_bl_results.biplot.save('final-data/08.rpca/baseline_phylo_rpca/biplot.qza')
phylo_rpca_bl_results.distance_matrix.save('final-data/08.rpca/baseline_phylo_rpca/distance-matrix.qza')
phylo_rpca_bl_results.counts_by_node_tree.save('final-data/08.rpca/baseline_phylo_rpca/counts_by_node_tree.qza')
phylo_rpca_bl_results.counts_by_node.save('final-data/08.rpca/baseline_phylo_rpca/counts_by_node.qza')
phylo_rpca_bl_results.t2t_taxonomy.save('final-data/08.rpca/baseline_phylo_rpca/t2t_taxonomy.qza')

'final-data/08.rpca/baseline_phylo_rpca/t2t_taxonomy.qza'

And then we'll do the rPCA for the full taxonomy.

In [22]:
deicode_results = \
    q2_gemelli.rpca(table_filt_q2, n_components=3)
phylo_rpca_results = \
    q2_gemelli.phylogenetic_rpca_with_taxonomy(table=table_filt_q2,
                                                  phylogeny=genome_tree_q2,
                                                  taxonomy=genome_taxa_q2.view(Metadata),
                                                  n_components=3
                                                 )

In [23]:
os.makedirs('final-data/05.rpca/full_rpca', exist_ok=True)
deicode_results.biplot.save('final-data/05.rpca/full_rpca/biplot.qza')
deicode_results.distance_matrix.save('final-data/05.rpca/full_rpca/distance-matrix.qza')

os.makedirs('final-data/08.rpca/phylo_rpca', exist_ok=True)
phylo_rpca_results.biplot.save('final-data/08.rpca/phylo_rpca/biplot.qza')
phylo_rpca_results.distance_matrix.save('final-data/08.rpca/phylo_rpca/distance-matrix.qza')
phylo_rpca_results.counts_by_node_tree.save('final-data/08.rpca/phylo_rpca/counts_by_node_tree.qza')
phylo_rpca_results.counts_by_node.save('final-data/08.rpca/phylo_rpca/counts_by_node.qza')
phylo_rpca_results.t2t_taxonomy.save('final-data/08.rpca/phylo_rpca/t2t_taxonomy.qza')

'final-data/08.rpca/phylo_rpca/t2t_taxonomy.qza'

# Antibiotic resistance 

In [24]:
gene_table = pd.read_csv('../data/Stag_output/amr_table.tsv',
                         sep='\t', dtype=str)
gene_table.set_index('sample-id', inplace=True)
gene_table = gene_table.T
gene_table_idx = gene_table.index.to_frame()[0].apply(lambda x: pd.Series(x.split('|')))
gene_table_idx.rename(columns={0: 'megares-id',
                               1: 'class', 
                               2: 'mechanism', 
                               3: 'group', 
                               4: 'gene', 
                               5: 'snp'},
                      inplace=True)
gene_table.index = pd.MultiIndex.from_frame(gene_table_idx)
gene_table.drop(['RequiresSNPConfirmation'], level='snp', inplace=True)
gene_table.reset_index(level='snp', inplace=True, drop=True)
gene_table.reset_index(level=['class', 'mechanism', 'group', 'gene'],
                       inplace=True)
gene_taxa = gene_table[['class', 'mechanism', 'group', 'gene']].copy()
gene_taxa2 = gene_taxa.apply(lambda x: ';'.join(x), axis=1)
gene_taxa2.index.set_names('Feature ID', inplace=True)
gene_taxa2.name = 'Taxon'

gene_table.drop(columns=['class', 'mechanism', 'group', 'gene'], inplace=True)
gene_table = gene_table.astype(float)
gene_table = gene_table.loc[gene_table.sum(axis=1) > 0].T.copy()
gene_table = gene_table.loc[working_meta.index].copy()

In [25]:
mgres_table_q2 = Artifact.import_data('FeatureTable[Frequency]', gene_table, pd.DataFrame)
mgres_taxa_q2 = Artifact.import_data('FeatureData[Taxonomy]', gene_taxa2, pd.Series)

In [26]:
gene_table.shape

(473, 1155)

In [27]:
gene_table_q2, = q2_taxa.collapse(mgres_table_q2, taxonomy=mgres_taxa_q2, level=3)
group_table_q2, = q2_taxa.collapse(mgres_table_q2, taxonomy=mgres_taxa_q2, level=2)

In [28]:
os.makedirs('final-data/03.amr-tables', exist_ok=True)
mgres_table_q2.save('final-data/03.amr-tables/megares_table_q2.qza')
mgres_taxa_q2.save('final-data/03.amr-tables/megares_taxa_q2.qza')
gene_table_q2.save('final-data/03.amr-tables/megares_gene_table_q2.qza')

'final-data/03.amr-tables/megares_gene_table_q2.qza'

Alpha Diveristy

In [29]:
resampled_gene_q2, = q2_boots.resample(gene_table_q2, 
                                      sampling_depth=2000,
                                      n=25,
                                      replacement=False)

In [30]:
%%capture
alpha_gene_q2 = {metric: {i: q2_diversity.alpha(table=table_q2, metric=metric).alpha_diversity
                          for i, table_q2 in resampled_gene_q2.items()}
                 for metric in ['observed_features', 'shannon', 'simpson']}
alpha_gene_avg_q2 = {
    metric: q2_boots.alpha_average(alphas, average_method='median').average_alpha_diversity
    for metric, alphas in alpha_gene_q2.items()
    }

In [31]:
os.makedirs('final-data/05.amr-alpha', exist_ok=True)
for metric, art_q2 in alpha_gene_avg_q2.items():
    art_q2.save(f'final-data/05.amr-alpha/gene_{metric}_2k_avg.qza')

In [32]:
%%capture
renames = {
    'shannon': 'shannon_entropy',
    'simpson': alpha_gene_avg_q2['simpson'].view(pd.Series).name,
}
alpha_baseline_diffs = {
    metric: q2_longitudinal.first_differences(
        working_meta_q2.merge(art.view(Metadata)),
        metric=renames.get(metric, metric),
        state_column='timepoint',
        individual_id_column='host_subject_id',
        baseline=0,
    ).first_differences
    for metric, art in alpha_gene_avg_q2.items()
}


for metric, alpha in alpha_baseline_diffs.items():
    alpha.save(f'final-data/05.amr-alpha/gene_{metric}_2K_avg_from_baseline.qza')

In [33]:
beta_gene_q2 =  {metric: {i: q2_diversity.beta(table=table_q2, metric=metric).distance_matrix
                          for i, table_q2 in resampled_gene_q2.items()}
                 for metric in ['braycurtis', 'jaccard']
                 }
beta_gene_avg_q2 = {
    metric: q2_boots.beta_average(dms_q2, 
                                  average_method='non-metric-median').average_distance_matrix
    for metric, dms_q2 in beta_gene_q2.items()
    }

In [34]:
os.makedirs('final-data/07.amr-beta', exist_ok=True)
for metric, art_q2 in beta_gene_avg_q2.items():
    art_q2.save(f'final-data/07.amr-beta/gene_{metric}_2k_avg.qza')

In [35]:
%%capture
from_baseline = {
    metric: q2_longitudinal.first_distances(
        distance_matrix=dm_q2,
        metadata=working_meta_q2,
        state_column='timepoint',
        individual_id_column='host_subject_id',
        baseline=0,
    ).first_distances
    for metric, dm_q2 in beta_gene_avg_q2.items()
}
for metric, dm_q2 in from_baseline.items():
    dm_q2.save(f'final-data/07.amr-beta/gene_{metric}_2k_avg_from_baseline.qza')